==================
وخلي ده معاك عشان هنحتاجه 

اتأكد من ال keys موجودة في قواعد البيانات ولو مشوجودة نضيفها وبنفس طريقة <LocalizedText/>

use this page as a refrence for translation and direction and keys' shape (<LocalizedText/>)
====================================

@page "/profile"
@attribute [Authorize]
@using Microsoft.EntityFrameworkCore
@using RubikCare.Application.Interfaces.Base
@using Rubikcare.Application.Interfaces.Localization
@using Rubikcare.Web.Data.Services.Helpers
@using System.Linq.Expressions
@using Rubikcare.Web.Components.Shared.Base
@using Rubikcare.Web.Data.Services.Session
@using System.Security.Claims

@inherits BaseFactoryCrudPage<UserProfile>

@inject AuthenticationStateProvider AuthenticationStateProvider
@inject UserSessionService UserSessionService
@inject IJSRuntime JS
@inject ExcelService excelService
@inject IImageService ImageService
@inject IGenericService<Area> AreaService
@inject ILocalizationService Localizer
@layout MainLayout
@rendermode InteractiveServer

<PageTitle><LocalizedText ResourceKey="ADMIN.USER_PROFILE.TITLE.PAGE" /> - RubikCare</PageTitle>

<!-- ========== ALERT SYSTEM ========== -->
@if (ShowAlert)
{
    <div class="alert-container alert-top-right">
        <div class="alert alert-@AlertType alert-dismissible fade show shadow-lg" role="alert">
            <div class="d-flex align-items-start">
                <i class="bi @GetAlertIcon() me-2 fs-5"></i>
                <div class="flex-grow-1">
                    @if (!string.IsNullOrEmpty(AlertTitle))
                    {
                        <h6 class="alert-heading mb-1">@AlertTitle</h6>
                    }
                    <p class="mb-0 small">@AlertMessage</p>
                </div>
                <button type="button" class="btn-close ms-2" @onclick="DismissAlert" aria-label="@Localizer.GetString("COMMON.CLOSE")"></button>
            </div>
            @if (AlertAutoClose)
            {
                <div class="progress mt-2" style="height: 3px;">
                    <div class="progress-bar bg-@AlertType"
                         style="width: 100%; animation: alertProgress @(AlertDuration)ms linear;">
                    </div>
                </div>
            }
        </div>
    </div>
}

<!-- ========== PROFILE MODAL ========== -->
@if (showProfileModal && editingUser != null)
{
    <div class="modal fade show d-block" style="background: rgba(0,0,0,0.5)">
        <div class="modal-dialog modal-lg">
            <div class="modal-content">
                <div class="modal-header bg-primary text-white">
                    <h5 class="modal-title">
                        <i class="bi bi-person"></i>
                        <LocalizedText ResourceKey="ADMIN.USER_PROFILE.TITLE.EDIT" />
                    </h5>
                    <button type="button" class="btn-close btn-close-white"
                            @onclick="CloseProfileModal"></button>
                </div>

                <div class="modal-body">
                    <!-- Profile Picture -->
                    <div class="text-center mb-4">
                        <ImageUploader @bind-ImageUrl="editingUser.ProfilePictureUrl"
                                       ImageType="profile"
                                       SubDirectory="profiles"
                                       PlaceholderText="@Localizer.GetString("ADMIN.USER_PROFILE.TOOLTIP.CHANGE_PICTURE")"
                                       ImageClass="img-fluid rounded-circle border shadow"
                                       CssClass="w-100" />
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.FULL_NAME_AR" /></label>
                                <input type="text" class="form-control"
                                       @bind="editingUser.FullNameAr"
                                       placeholder="@Localizer.GetString("ADMIN.USER_PROFILE.PLACEHOLDER.FULL_NAME_AR")" />
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.FULL_NAME_EN" /></label>
                                <input type="text" class="form-control"
                                       @bind="editingUser.FullNameEn"
                                       placeholder="@Localizer.GetString("ADMIN.USER_PROFILE.PLACEHOLDER.FULL_NAME_EN")" />
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.FIRST_NAME" /> *</label>
                                <input type="text" class="form-control"
                                       @bind="editingUser.FirstName" />
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.LAST_NAME" /> *</label>
                                <input type="text" class="form-control"
                                       @bind="editingUser.LastName" />
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.PHONE" /> *</label>
                                <input type="text" class="form-control"
                                       @bind="editingUser.PhoneNumber" />
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.EMAIL" /></label>
                                <input type="email" class="form-control"
                                       @bind="editingUser.Email" />
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.NATIONAL_ID" /></label>
                                <input type="text" class="form-control"
                                       @bind="editingUser.NationalID" />
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.GENDER" /></label>
                                <select class="form-select" @bind="editingUser.Gender">
                                    <option value=""><LocalizedText ResourceKey="ADMIN.USER_PROFILE.PLACEHOLDER.SELECT_GENDER" /></option>
                                    <option value="M"><LocalizedText ResourceKey="COMMON.GENDER.MALE" /></option>
                                    <option value="F"><LocalizedText ResourceKey="COMMON.GENDER.FEMALE" /></option>
                                </select>
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.AREA" /></label>
                                <select class="form-select" @bind="editingUser.AreaID">
                                    <option value=""><LocalizedText ResourceKey="ADMIN.USER_PROFILE.PLACEHOLDER.SELECT_AREA" /></option>
                                    @foreach (var area in allAreas.OrderBy(a => a.AreaNameAr))
                                    {
                                        <option value="@area.AreaID">@area.AreaNameAr</option>
                                    }
                                </select>
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="mb-3">
                                <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.DOB" /></label>
                                <input type="date" class="form-control"
                                       @bind="editingUser.DateOfBirth" />
                            </div>
                        </div>
                    </div>

                    <div class="mb-3">
                        <label class="form-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.ADDRESS" /></label>
                        <textarea class="form-control" rows="2"
                                  @bind="editingUser.Address"></textarea>
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="form-check form-switch mt-4 pt-2">
                                <input class="form-check-input" type="checkbox"
                                       @bind="editingUser.IsActive" />
                                <label class="form-check-label"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.ACCOUNT_STATUS" /></label>
                            </div>
                        </div>
                    </div>
                </div>

                <div class="modal-footer">
                    <button type="button" class="btn btn-secondary"
                            @onclick="CloseProfileModal">
                        <LocalizedText ResourceKey="ADMIN.USER_PROFILE.BUTTON.CANCEL" />
                    </button>
                    <button type="button" class="btn btn-primary"
                            @onclick="SaveProfile">
                        <LocalizedText ResourceKey="ADMIN.USER_PROFILE.BUTTON.SAVE_CHANGES" />
                    </button>
                </div>
            </div>
        </div>
    </div>
}

<!-- ========== MAIN CONTENT ========== -->
<div class="rubik-main-content" style="margin-top: 60px; padding: 20px;">
    <div class="container-fluid animate__animated animate__fadeIn">

        <!-- ========== PAGE HEADER ========== -->
        <div class="d-flex justify-content-between align-items-center mb-4">
            <div>
                <h2 class="text-rubik-primary fw-bold">
                    <i class="bi bi-person-circle me-2"></i><LocalizedText ResourceKey="ADMIN.USER_PROFILE.TITLE.PAGE" />
                </h2>
                <nav aria-label="breadcrumb">
                    <ol class="breadcrumb mb-0">
                        <li class="breadcrumb-item"><a href="/"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.BREADCRUMB.HOME" /></a></li>
                        <li class="breadcrumb-item active"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.BREADCRUMB.CURRENT" /></li>
                    </ol>
                </nav>
            </div>

            <div class="d-flex gap-2">
                <RubikButton Icon="bi-pencil"
                             ColorClass="btn-primary"
                             OnClick="() => OpenEditModal(currentUser)"
                             CssClass="px-4">
                    <LocalizedText ResourceKey="ADMIN.USER_PROFILE.BUTTON.EDIT" />
                </RubikButton>
            </div>
        </div>

        <!-- ========== LOADING STATE ========== -->
        @if (IsLoading)
        {
            <div class="text-center py-5 shadow-sm bg-white rounded border">
                <div class="spinner-border text-primary" role="status"></div>
                <p class="mt-3 text-muted fw-bold"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.MESSAGE.LOADING" /></p>
            </div>
        }
        else if (currentUser == null)
        {
            <div class="text-center py-5 shadow-sm bg-white rounded border">
                <i class="bi bi-person-x fs-1 text-muted mb-3"></i>
                <h5 class="text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.MESSAGE.NO_PROFILE" /></h5>
                <p class="text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.MESSAGE.CONTACT_SUPPORT" /></p>
            </div>
        }
        else
        {
            <div class="row">
                <!-- LEFT COLUMN -->
                <div class="col-md-4 mb-4">
                    <div class="card shadow-sm border-0 h-100">
                        <div class="card-body text-center">
                            <div class="mb-4">
                                @if (!string.IsNullOrEmpty(currentUser.ProfilePictureUrl))
                                {
                                    <img src="@currentUser.ProfilePictureUrl"
                                         class="img-fluid rounded-circle border shadow"
                                         style="width: 200px; height: 200px; object-fit: cover;"
                                         alt="@currentUser.GetFullName("ar")"
                                         onerror="this.src='/uploads/profiles/default-avatar.png'" />
                                }
                                else
                                {
                                    <div class="rounded-circle border d-inline-flex align-items-center justify-content-center bg-light"
                                         style="width: 200px; height: 200px;">
                                        <i class="bi bi-person-fill text-muted" style="font-size: 5rem;"></i>
                                    </div>
                                }
                            </div>

                            <h4 class="text-primary">@currentUser.GetFullName("ar")</h4>
                            @if (!string.IsNullOrEmpty(currentUser.GetFullName("en")))
                            {
                                <p class="text-muted">@currentUser.GetFullName("en")</p>
                            }

                            <div class="mt-3">
                                <span class="badge @(currentUser.IsActive ? "bg-success" : "bg-secondary")">
                                    <i class="bi @(currentUser.IsActive ? "bi-check-circle" : "bi-x-circle") me-1"></i>
                                    @(currentUser.IsActive? Localizer.GetString("ADMIN.USER_PROFILE.LABEL.ACTIVE") : Localizer.GetString("ADMIN.USER_PROFILE.LABEL.INACTIVE"))
                                </span>
                            </div>

                            <div class="mt-4 pt-3 border-top">
                                <div class="row text-center">
                                    <div class="col-6 border-end">
                                        <small class="text-muted d-block"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.MEMBERSHIP_ID" /></small>
                                        <strong class="text-primary">#@currentUser.UserProfileID</strong>
                                    </div>
                                    <div class="col-6">
                                        <small class="text-muted d-block"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.DOB" /></small>
                                        <strong>@(currentUser.DateOfBirth?.ToString("dd/MM/yyyy") ?? "-")</strong>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>

                <!-- RIGHT COLUMN -->
                <div class="col-md-8">
                    <div class="card shadow-sm border-0 h-100">
                        <div class="card-body">
                            <!-- Personal Information -->
                            <h5 class="text-primary border-bottom pb-2 mb-3">
                                <i class="bi bi-person-lines-fill me-2"></i><LocalizedText ResourceKey="ADMIN.USER_PROFILE.SECTION.PERSONAL_INFO" />
                            </h5>

                            <div class="row">
                                <div class="col-md-6">
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.FULL_NAME_AR" /></label>
                                        <p class="form-control-plaintext fw-bold">@currentUser.FullNameAr</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.FULL_NAME_EN" /></label>
                                        <p class="form-control-plaintext">@currentUser.FullNameEn</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.FIRST_NAME" /></label>
                                        <p class="form-control-plaintext">@currentUser.FirstName</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.LAST_NAME" /></label>
                                        <p class="form-control-plaintext">@currentUser.LastName</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.GENDER" /></label>
                                        <p class="form-control-plaintext">
                                            @{
                                                var genderText = currentUser.Gender switch
                                                {
                                                                "M" => Localizer.GetString("COMMON.GENDER.MALE"),
                                                                "F" => Localizer.GetString("COMMON.GENDER.FEMALE"),
                                                    _ => Localizer.GetString("COMMON.GENDER.UNSPECIFIED")
                                                };
                                            }
                                            @genderText
                                        </p>
                                    </div>
                                </div>

                                <div class="col-md-6">
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.PHONE" /></label>
                                        <p class="form-control-plaintext">@currentUser.PhoneNumber</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.EMAIL" /></label>
                                        <p class="form-control-plaintext">@currentUser.Email</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.NATIONAL_ID" /></label>
                                        <p class="form-control-plaintext">@currentUser.NationalID</p>
                                    </div>
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.DOB" /></label>
                                        <p class="form-control-plaintext">
                                            @(currentUser.DateOfBirth?.ToString("dd/MM/yyyy") ?? "-")
                                            @if (currentUser.DateOfBirth.HasValue)
                                            {
                                                <span class="text-muted ms-2">
                                                    (@CalculateAge(currentUser.DateOfBirth.Value) <LocalizedText ResourceKey="COMMON.YEARS" />)
                                                </span>
                                            }
                                        </p>
                                    </div>
                                </div>
                            </div>

                            <!-- Contact Information -->
                            <h5 class="text-primary border-bottom pb-2 mb-3 mt-4">
                                <i class="bi bi-geo-alt-fill me-2"></i><LocalizedText ResourceKey="ADMIN.USER_PROFILE.SECTION.CONTACT_INFO" />
                            </h5>

                            <div class="row">
                                <div class="col-md-6">
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.AREA" /></label>
                                        <p class="form-control-plaintext">@currentUser?.Area?.AreaNameAr</p>
                                    </div>
                                </div>
                                <div class="col-md-6">
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.ADDRESS" /></label>
                                        <p class="form-control-plaintext">@currentUser?.Address</p>
                                    </div>
                                </div>
                                @if (currentUser != null && currentUser.Latitude.HasValue && currentUser.Longitude.HasValue)
                                {
                                    <div class="col-md-6">
                                        <div class="mb-3">
                                            <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.COORDINATES" /></label>
                                            <p class="form-control-plaintext">
                                                @currentUser.Latitude, @currentUser.Longitude
                                            </p>
                                        </div>
                                    </div>
                                }
                            </div>

                            <!-- System Information -->
                            <h5 class="text-primary border-bottom pb-2 mb-3 mt-4">
                                <i class="bi bi-info-circle me-2"></i><LocalizedText ResourceKey="ADMIN.USER_PROFILE.SECTION.SYSTEM_INFO" />
                            </h5>

                            <div class="row">
                                <div class="col-md-6">
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.CREATED_DATE" /></label>
                                        <p class="form-control-plaintext">@currentUser?.CreatedDate.ToString("dd/MM/yyyy HH:mm")</p>
                                    </div>
                                </div>
                                <div class="col-md-6">
                                    <div class="mb-3">
                                        <label class="form-label text-muted"><LocalizedText ResourceKey="ADMIN.USER_PROFILE.LABEL.LAST_UPDATED" /></label>
                                        <p class="form-control-plaintext">@currentUser?.LastModifiedDate.ToString("dd/MM/yyyy HH:mm")</p>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
            </div>
        }
    </div>
</div>

@code {
    private UserProfile? currentUser = null;
    private UserProfile? editingUser = null;
    private bool showProfileModal = false;
    private List<Area> allAreas = new();

    protected override async Task LoadDataAsync()
    {
        try
        {
            IsLoading = true;
            ErrorMessage = string.Empty;
            Console.WriteLine($"[MyProfile] Loading data...");

            // Load areas for dropdown
            allAreas = (await AreaService.GetAllAsync()).ToList();
            Console.WriteLine($"[MyProfile] Loaded {allAreas.Count} areas");

            // ========== الحصول على المستخدم الحالي من نظام المصادقة ==========
            var authState = await AuthenticationStateProvider.GetAuthenticationStateAsync();
            var user = authState.User;

            if (user.Identity?.IsAuthenticated == true)
            {
                var userId = user.FindFirst(ClaimTypes.NameIdentifier)?.Value;
                Console.WriteLine($"[MyProfile] User ID from claims: {userId}");

                if (!string.IsNullOrEmpty(userId))
                {
                    // استخدام GetUserBasicInfoAsync للحصول على UserProfileId
                    var basicInfo = await UserSessionService.GetUserBasicInfoAsync(userId);

                    if (basicInfo != null && basicInfo.UserProfileId > 0)
                    {
                        var currentUserId = basicInfo.UserProfileId;
                        Console.WriteLine($"[MyProfile] UserProfileId from session: {currentUserId}");

                        // Load user with relations
                        currentUser = await DbFactory.ExecuteReadOnlyAsync(async context =>
                        {
                            var dbContext = (DbContext)context; // ⭐ إضافة هذا السطر
                            return await dbContext.Set<UserProfile>()
                                .Include(u => u.Area)
                                .ThenInclude(a => a.City)
                                .ThenInclude(c => c.Country)
                                .FirstOrDefaultAsync(u => u.UserProfileID == currentUserId);
                        });

                        Console.WriteLine($"[MyProfile] User loaded: {currentUser != null}");
                    }
                    else
                    {
                        Console.WriteLine($"[MyProfile] No UserProfileId found for user {userId}");
                    }
                }
            }
            else
            {
                Console.WriteLine($"[MyProfile] User is not authenticated");
            }

            // إذا لم يتم العثور على مستخدم، نترك currentUser = null
            // وهذا سيعرض رسالة "لم يتم العثور على الملف الشخصي"

            Items = currentUser != null
                ? new List<UserProfile> { currentUser }
                : new List<UserProfile>();

            Console.WriteLine($"[MyProfile] Data loaded successfully");
        }
        catch (Exception ex)
        {
            ErrorMessage = $"{Localizer.GetString("ADMIN.USER_PROFILE.ALERT.LOAD_FAILED")}: {ex.Message}";
            ShowErrorAlert(ErrorMessage);
            Console.WriteLine($"[ERROR] LoadDataAsync: {ex.Message}");
        }
        finally
        {
            IsLoading = false;
            StateHasChanged();
        }
    }

    private void OpenEditModal(UserProfile user)
    {
        try
        {
            Console.WriteLine($"[MyProfile] Opening edit modal");

            editingUser = new UserProfile
            {
                UserProfileID = user.UserProfileID,
                FirstName = user.FirstName ?? "",
                LastName = user.LastName ?? "",
                FullNameAr = user.FullNameAr,
                FullNameEn = user.FullNameEn,
                NationalID = user.NationalID,
                Email = user.Email,
                PhoneNumber = user.PhoneNumber ?? "",
                DateOfBirth = user.DateOfBirth,
                Gender = user.Gender,
                ProfilePictureUrl = user.ProfilePictureUrl,
                IsActive = user.IsActive,
                Address = user.Address,
                AreaID = user.AreaID,
                Latitude = user.Latitude,
                Longitude = user.Longitude
            };

            showProfileModal = true;
            StateHasChanged();
        }
        catch (Exception ex)
        {
            ShowErrorAlert($"{Localizer.GetString("COMMON.ERROR")}: {ex.Message}");
            Console.WriteLine($"[ERROR] OpenEditModal: {ex.Message}");
        }
    }

    private async Task CloseProfileModal()
    {
        showProfileModal = false;
        editingUser = null;
        await LoadDataAsync();
    }

    private async Task SaveProfile()
    {
        if (editingUser == null) return;

        try
        {
            // Validation
            if (string.IsNullOrWhiteSpace(editingUser.FirstName))
            {
                ShowErrorAlert(Localizer.GetString("ADMIN.USER_PROFILE.ALERT.VALIDATION_FIRST_NAME"));
                return;
            }

            if (string.IsNullOrWhiteSpace(editingUser.LastName))
            {
                ShowErrorAlert(Localizer.GetString("ADMIN.USER_PROFILE.ALERT.VALIDATION_LAST_NAME"));
                return;
            }

            if (string.IsNullOrWhiteSpace(editingUser.PhoneNumber))
            {
                ShowErrorAlert(Localizer.GetString("ADMIN.USER_PROFILE.ALERT.VALIDATION_PHONE"));
                return;
            }

            // Save using DbFactory
            await DbFactory.ExecuteWithNewContextAsync(async context =>
            {
                var dbContext = (DbContext)context; // ⭐ إضافة هذا السطر
                var userProfilesSet = dbContext.Set<UserProfile>(); // ⭐ استخدام Set<T>

                var existing = await userProfilesSet
                    .FindAsync(editingUser.UserProfileID);

                if (existing != null)
                {
                    // Update existing
                    dbContext.Entry(existing).CurrentValues.SetValues(editingUser);
                    userProfilesSet.Update(existing);
                }
                else
                {
                    // Add new
                    await userProfilesSet.AddAsync(editingUser); // ⭐ await مع AddAsync
                }

                await dbContext.SaveChangesAsync(); // ⭐ حفظ على dbContext
            });

            // Reload and close modal
            await LoadDataAsync();
            showProfileModal = false;
            editingUser = null;

            ShowSuccessAlert(Localizer.GetString("ADMIN.USER_PROFILE.ALERT.SAVE_SUCCESS"));
            Console.WriteLine($"[MyProfile] Save successful");
        }
        catch (Exception ex)
        {
            ShowErrorAlert($"{Localizer.GetString("COMMON.ERROR")}: {ex.Message}");
            Console.WriteLine($"[ERROR] SaveProfile: {ex.Message}");
        }
    }

    private int CalculateAge(DateTime birthDate)
    {
        var today = DateTime.Today;
        var age = today.Year - birthDate.Year;
        if (birthDate.Date > today.AddYears(-age)) age--;
        return age;
    }
}